In [ ]:
%useLatestDescriptors
%use dataframe(1.0.0-Beta5n)
%use kandy
%use serialization

In [ ]:
@file:Repository("https://repo.gradle.org/gradle/libs-releases") //
@file:DependsOn("org.gradle:gradle-tooling-api:9.6.0")

import org.gradle.tooling.GradleConnector
import java.io.File

GradleConnector.newConnector().forProjectDirectory(File("./../../")).connect().use { connection ->
    connection.newBuild()
            .forTasks(
                "clean",
                "reverseBytesBenchmark", "reverseBitsBenchmark",
                "singleBitsBenchmark", "multipleBitsBenchmark"
            )
            .setStandardOutput(System.out)
            .setStandardError(System.err)
            .run()
}

In [5]:
@file:OptIn(ExperimentalSerializationApi::class)

import kotlinx.serialization.ExperimentalSerializationApi
import kotlinx.serialization.SerialName
import kotlinx.serialization.Serializable
import kotlinx.serialization.json.Json
import kotlinx.serialization.json.decodeFromStream
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.bars
import java.nio.file.Path
import kotlin.io.path.PathWalkOption
import kotlin.io.path.extension
import kotlin.io.path.inputStream
import kotlin.io.path.name
import kotlin.io.path.nameWithoutExtension
import kotlin.io.path.walk

fun String.suffixSimilarity(other: String): Double {
    val a = this.reversed()
    val b = other.reversed()
    val maxLen = maxOf(a.length, b.length)
    if (maxLen == 0) return 1.0
    var match = 0
    for (i in 0 until minOf(a.length, b.length)) {
        if (a[i] == b[i]) match++ else break
    }
    return match.toDouble() / maxLen
}

inline fun <T> List<T>.groupBySimilarityChain(crossinline selector: (T) -> String): List<T> {
    if (isEmpty()) return emptyList()
    val unused = toMutableSet()
    val result = ArrayList<T>()
    var current = unused.first()
    result += current
    unused -= current
    while (unused.isNotEmpty()) {
        val next = unused.maxBy { selector(it).suffixSimilarity(selector(current)) }
        result += next
        unused -= next
        current = next
    }
    return result
}

@Serializable
data class PrimaryMetric(
    val score: Double,
    val scoreError: Double,
    val scoreUnit: String
)

@Serializable
data class Benchmark(
    @SerialName("benchmark") val name: String,
    val warmupIterations: Int,
    val measurementIterations: Int,
    val primaryMetric: PrimaryMetric
) {
    inline val cleanName: String
        get() = name.substringBeforeLast('.').substringAfterLast('.')
}

val json: Json = Json { ignoreUnknownKeys = true }

Path.of("./../build/reports/benchmarks")
        .walk()
        .filter { filePath -> filePath.extension == "json" }
        .map { filePath ->
            filePath.inputStream().use { stream ->
                filePath to json.decodeFromStream<List<Benchmark>>(stream)
            }
        }.map { (filePath, report) ->
            val sortedReport = report.groupBySimilarityChain(Benchmark::name)
            val plotName = filePath.parent.parent.name
            val platform = filePath.nameWithoutExtension
            val df = dataFrameOf(
                "name" to sortedReport.map(Benchmark::cleanName),
                "score" to sortedReport.map { benchmark -> benchmark.primaryMetric.score },
                "score_min" to sortedReport.map { benchmark -> benchmark.primaryMetric.score - benchmark.primaryMetric.scoreError },
                "score_max" to sortedReport.map { benchmark -> benchmark.primaryMetric.score + benchmark.primaryMetric.scoreError }
            )
            val plot = plot(df) {
                layout {
                    size = 1200 to 800
                    title = "$plotName ($platform)" // Parent of parent is suite name
                    xAxisLabel = "Implementation"
                    yAxisLabel = if("reverse" in plotName.lowercase()) "ops/s" else "MiB/s"
                    theme = Theme.DARCULA
                }
                bars {
                    x("name")
                    y("score") {
                        axis {
                            breaks(format = ".2f")
                        }
                    }
                    fillColor("name")
                }
                errorBars {
                    x("name")
                    yMin("score_min")
                    yMax("score_max")
                }
            }
            plot.save("./../../docs/${plotName}_$platform.png")
            plot
        }.forEach(::DISPLAY)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="RO0h6f" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("RO0h6f");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"multipleBits (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[1.9444664988371883,0.6446314525713902,7.225606442971852,0.960422658619583],
"name":["CommonReadMultipleBitsBenchmark","KarbideReadMultipleBitsBenchmark","CommonWriteMultipleBitsBenchmark","KarbideWriteMultipleBitsBenchmark"],
"score_max":[2.060681851140778,0.6664265720433966,7.542193223267383,0.9657690407780544],
"score_min":[1.8282511465335984,0.6228363330993838,6.909019662676321,0.9550762764611116]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"143"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 0.50 
 
 
 
 
 
 
 1.00 
 
 
 
 
 
 
 1.50 
 
 
 
 
 
 
 2.00 
 
 
 
 
 
 
 2.50 
 
 
 
 
 
 
 3.00 
 
 
 
 
 
 
 3.50 
 
 
 
 
 
 
 4.00 
 
 
 
 
 
 
 4.50 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 5.50 
 
 
 
 
 
 
 6.00 
 
 
 
 
 
 
 6.50 
 
 
 
 
 
 
 7.00 
 
 
 
 
 
 
 7.50 
 
 
 
 
 
 
 
 
 multipleBits (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="vBKH4A" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("vBKH4A");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"multipleBits (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[20.23923700458876,7.185275745053998,24.597905549867313,12.763562832148015],
"name":["CommonReadMultipleBitsBenchmark","KarbideReadMultipleBitsBenchmark","CommonWriteMultipleBitsBenchmark","KarbideWriteMultipleBitsBenchmark"],
"score_max":[21.45853957268172,7.523519726154831,25.81000133456866,12.944331489342392],
"score_min":[19.0199344364958,6.847031763953165,23.385809765165966,12.582794174953637]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"147"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 2.00 
 
 
 
 
 
 
 4.00 
 
 
 
 
 
 
 6.00 
 
 
 
 
 
 
 8.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 12.00 
 
 
 
 
 
 
 14.00 
 
 
 
 
 
 
 16.00 
 
 
 
 
 
 
 18.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 22.00 
 
 
 
 
 
 
 24.00 
 
 
 
 
 
 
 26.00 
 
 
 
 
 
 
 
 
 multipleBits (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonR

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ZqFmqM" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("ZqFmqM");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"multipleBits (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[21.219248034880295,6.886411346655715,25.745836101220817,13.166375624684147],
"name":["CommonReadMultipleBitsBenchmark","KarbideReadMultipleBitsBenchmark","CommonWriteMultipleBitsBenchmark","KarbideWriteMultipleBitsBenchmark"],
"score_max":[23.645465430925505,7.151257057985704,26.42277981821166,13.424447337634618],
"score_min":[18.793030638835084,6.621565635325725,25.068892384229976,12.908303911733675]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"151"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 2.00 
 
 
 
 
 
 
 4.00 
 
 
 
 
 
 
 6.00 
 
 
 
 
 
 
 8.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 12.00 
 
 
 
 
 
 
 14.00 
 
 
 
 
 
 
 16.00 
 
 
 
 
 
 
 18.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 22.00 
 
 
 
 
 
 
 24.00 
 
 
 
 
 
 
 26.00 
 
 
 
 
 
 
 
 
 multipleBits (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonR

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="hoz2BV" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("hoz2BV");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"multipleBits (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[17.24750850983184,7.927801047710832,19.98430438790314,12.04661187846192],
"name":["CommonReadMultipleBitsBenchmark","KarbideReadMultipleBitsBenchmark","CommonWriteMultipleBitsBenchmark","KarbideWriteMultipleBitsBenchmark"],
"score_max":[18.204266491794733,8.204734789986839,20.492695127445124,12.532176460510847],
"score_min":[16.290750527868944,7.650867305434826,19.475913648361157,11.561047296412994]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"155"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 2.00 
 
 
 
 
 
 
 4.00 
 
 
 
 
 
 
 6.00 
 
 
 
 
 
 
 8.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 12.00 
 
 
 
 
 
 
 14.00 
 
 
 
 
 
 
 16.00 
 
 
 
 
 
 
 18.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 
 
 multipleBits (linuxX64) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmar 
 
 
 k 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideR

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="yjIg1W" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("yjIg1W");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"multipleBits (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[45.594755258722024,22.435500525523622,25.694793644666674,22.82068782954034],
"name":["CommonReadMultipleBitsBenchmark","KarbideReadMultipleBitsBenchmark","CommonWriteMultipleBitsBenchmark","KarbideWriteMultipleBitsBenchmark"],
"score_max":[52.47230104661415,22.9923986553358,26.129912802980183,23.66679710607142],
"score_min":[38.7172094708299,21.878602395711443,25.259674486353166,21.97457855300926]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"159"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteMultipleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 55.00 
 
 
 
 
 
 
 
 
 multipleBits (jvm) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadMultipleBitsBenchmar 
 
 
 k 
 
 
 
 
 
 
 
 
 
 
 


...

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="CAcNRA" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("CAcNRA");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[1.8644727466064388,0.5903700169423152,9.441528577373616,0.331366125096576],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[1.967744777270857,0.613940749074181,10.030912184920508,0.33464451165705306],
"score_min":[1.7612007159420207,0.5667992848104494,8.852144969826723,0.3280877385360989]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"203"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 1.00 
 
 
 
 
 
 
 2.00 
 
 
 
 
 
 
 3.00 
 
 
 
 
 
 
 4.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 6.00 
 
 
 
 
 
 
 7.00 
 
 
 
 
 
 
 8.00 
 
 
 
 
 
 
 9.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 
 
 singleBits (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="T7jkOr" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("T7jkOr");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[30.02705939403404,5.10320483741377,44.74768316677105,10.5387632997188],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[33.0027398799574,5.270051293515338,45.470596406828655,10.818220017866842],
"score_min":[27.051378908110674,4.9363583813122025,44.02476992671344,10.259306581570758]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"207"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 
 
 singleBits (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 C

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="OO51z7" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("OO51z7");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[32.69800999687996,5.022076094374244,44.70319473687316,10.978160681069305],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[35.55548528402563,5.240435847037928,45.20547993549832,11.218903802341885],
"score_min":[29.840534709734296,4.803716341710559,44.200909538248,10.737417559796725]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"211"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 
 
 singleBits (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Commo

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="EtEHIr" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("EtEHIr");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[29.677187087141817,9.489280945327248,55.61401314796508,6.097960846621984],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[30.352038058914605,9.808363464677013,56.404857293002166,6.181256937577235],
"score_min":[29.00233611536903,9.170198425977484,54.823169002928,6.0146647556667325]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"215"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 55.00 
 
 
 
 
 
 
 
 
 singleBits (linuxX64) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSin

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="sc77b3" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("sc77b3");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[222.30407298000773,27.449415493199638,51.92159516587585,17.218767577206165],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[230.22967705173622,28.922809410337543,55.05686975478638,17.76356832767438],
"score_min":[214.37846890827925,25.976021576061733,48.786320576965316,16.67396682673795]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"219"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 80.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 120.00 
 
 
 
 
 
 
 140.00 
 
 
 
 
 
 
 160.00 
 
 
 
 
 
 
 180.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 220.00 
 
 
 
 
 
 
 240.00 
 
 
 
 
 
 
 
 
 singleBits (jvm) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
